In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'SSTv2'
name_model = 'claude-4.5'
mode = 'few'
seed = 1
part = 2

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post18/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post19/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/ANTHROPIC_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.9/397.9 kB 13.2 MB/s eta 0:00:00


In [10]:
import json
import requests
import pandas as pd
from anthropic import Anthropic

In [11]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [12]:
ANTHROPIC_API_KEY = credentials["ANTHROPIC_API_KEY"]

In [13]:
client = Anthropic(api_key=ANTHROPIC_API_KEY)

# 3. Functions

In [14]:
def few_shot_prompt(text):

    ex_neg_1 = "we do n't even like their characters ."

    ex_pos_1 = (
        "of those unassuming films that sneaks up on you and stays with you "
        "long after you have left the theater"
    )

    intro = (
        "Classify the sentiment of the following movie sentence from the "
        "Stanford Sentiment Treebank v2 (SST-2) dataset.\n"
        "Respond only with a single digit: 0 if the sentiment is negative, "
        "1 if the sentiment is positive.\n"
        "Return only the digit (no words, no punctuation).\n"
    )

    few_shots = (
        "\nHere are some examples:\n\n"
        "Example 1:\n"
        f"Sentence: \"{ex_pos_1}\"\n"
        "Label: 1\n\n"
        "Example 2:\n"
        f"Sentence: \"{ex_neg_1}\"\n"
        "Label: 0\n\n"
    )

    target = (
        "Now classify the following sentence:\n"
        f"Sentence: \"{text}\"\n"
        "Label:"
    )

    return intro + few_shots + target

In [15]:
def predict_label(text, prompt):

    try:

      start_time = time.perf_counter()
      first_token_time = None
      output_text = ""
      usage = None

      with client.messages.stream(
          model="claude-sonnet-4-5-20250929",
          max_tokens=16,
          temperature=0.0,
          messages=[{"role": "user", "content": prompt}],
      ) as stream:
          for event in stream:
              if event.type == "content_block_delta":
                  delta_text = getattr(event.delta, "text", None)
                  if delta_text:
                      if first_token_time is None:
                          first_token_time = time.perf_counter()
                      output_text += delta_text

          final_msg = stream.get_final_message()
          usage = getattr(final_msg, "usage", None)

      end_time = time.perf_counter()

      return {
          "prediction": output_text.strip(),
          "latency_ms": (end_time - start_time) * 1000,
          "ttft_ms": ((first_token_time - start_time) * 1000) if first_token_time else None,
          "input_tokens": getattr(usage, "input_tokens", None) if usage else None,
          "output_tokens": getattr(usage, "output_tokens", None) if usage else None,
      }

    except:

      return {
          "prediction": '-1',
          "latency_ms": '-',
          "ttft_ms": '-',
          "input_tokens": '-',
          "output_tokens": '-',
      }


# 4. Load Dataset

In [16]:
df = pd.read_csv(path_open)

In [17]:
df.shape

(2021, 66)

In [18]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [19]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = few_shot_prompt(text)
  output = predict_label(text, prompt)

  try:
    pred_label.append(int(output['prediction']))
    pred_latency.append(output['latency_ms'])
    pred_ttft.append(output['ttft_ms'])
    pred_input_tokens.append(output['input_tokens'])
    pred_output_tokens.append(output['output_tokens'])

  except:
    pred_label.append(-1)
    pred_latency.append('-')
    pred_ttft.append('-')
    pred_input_tokens.append('-')
    pred_output_tokens.append('-')

  time.sleep(1)

  if (i % 10) == 0:
    print(i)

0


10


20


30


40


...

1980


1990


2000


2010


2020


In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [21]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
claude-4.5-few-seed-1-label,
1,1062
0,959


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,claude-4.5-few-seed-1-latency
count,2021.000000
mean,1489.739799
std,449.306048
min,807.162761
25%,1314.497435
50%,1408.952382
75%,1549.080911
max,7403.324173


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,claude-4.5-few-seed-1-ttft
count,2021.000000
mean,1434.267126
std,443.763159
min,750.029037
25%,1267.716420
50%,1360.600182
75%,1489.946285
max,7382.418612


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,claude-4.5-few-seed-1-input-tokens
count,2021.000000
mean,174.637803
std,9.527367
min,164.000000
25%,167.000000
50%,171.000000
75%,179.000000
max,217.000000


In [25]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,claude-4.5-few-seed-1-output-tokens
count,2021.0
mean,5.0
std,0.0
min,5.0
25%,5.0
50%,5.0
75%,5.0
max,5.0


# 5. Save Dataset

In [26]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [27]:
end_notebook = time.time()

In [28]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 1h 26m 5.14s
